# Modelado de Fatiga con LSTM (Long Short-Term Memory) Personalizada

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada, y la implementación paso a paso de una red **LSTM implementada manualmente en PyTorch** (sin recurrir al módulo `nn.LSTM` estándar) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

La arquitectura **Long Short-Term Memory (LSTM)** fue propuesta originalmente por Sepp Hochreiter y Jürgen Schmidhuber en 1997 para solucionar el problema del desvanecimiento y la explosión del gradiente en redes neuronales recurrentes (RNN) tradicionales al procesar dependencias de largo plazo.

El núcleo de una LSTM es el estado de la celda ($c_t$), que actúa como una cinta transportadora que permite el flujo de información con cambios lineales menores a lo largo del tiempo. Este flujo es controlado por estructuras especializadas llamadas **compuertas (gates)**, las cuales deciden qué información retener, actualizar o descartar.

### Formulación Matemática

Para un paso de tiempo $t$, dada la entrada actual $x_t$, el estado oculto previo $h_{t-1}$ y el estado de la celda previo $c_{t-1}$, la celda calcula:

1. **Compuerta de Olvido (Forget Gate):** Determina qué proporción del estado de celda anterior se descarta.
   $$f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)$$

2. **Compuerta de Entrada (Input Gate):** Controla qué nueva información se almacenará en el estado de la celda.
   $$i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)$$

3. **Candidato de Estado de Celda (Candidate Cell State):** Genera nuevos valores candidatos que podrían agregarse al estado.
   $$\tilde{c}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$$

4. **Actualización del Estado de Celda (Cell State Update):** Combina el olvido de la memoria pasada y la adición del nuevo candidato.
   $$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

5. **Compuerta de Salida (Output Gate):** Decide qué partes del estado de la celda se enviarán al estado oculto.
   $$o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)$$

6. **Estado Oculto de la Celda (Hidden State Update):** Produce la salida visible de la celda y se transmite al siguiente paso.
   $$h_t = o_t \odot \tanh(c_t)$$

Donde:
* $\sigma(z) = \frac{1}{1 + e^{-z}}$ es la función sigmoide.
* $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ es la tangente hiperbólica.
* $\odot$ representa el producto de Hadamard (multiplicación elemento a elemento).
* $W_* \in \mathbb{R}^{d_h \times d_x}$, $U_* \in \mathbb{R}^{d_h \times d_h}$ y $b_* \in \mathbb{R}^{d_h}$ son los pesos y sesgos de la red.

---

### Diagrama de Flujo de la Celda (Mermaid)

```mermaid
graph TD
    subgraph "Celda Custom LSTM (Paso t)"
        xt["Entrada actual: x_t"]
        h_prev["Estado oculto anterior: h_{t-1}"]
        c_prev["Estado de celda anterior: c_{t-1}"]

        f_t["Forget Gate: f_t = σ(W_f x_t + U_f h_{t-1} + b_f)"]
        i_t["Input Gate: i_t = σ(W_i x_t + U_i h_{t-1} + b_i)"]
        c_tilde["Candidato Celda: c̃_t = tanh(W_c x_t + U_c h_{t-1} + b_c)"]
        o_t["Output Gate: o_t = σ(W_o x_t + U_o h_{t-1} + b_o)"]

        c_update["Actualización Celda: c_t = (f_t ⊙ c_{t-1}) + (i_t ⊙ c̃_t)"]
        h_update["Actualización Oculta: h_t = o_t ⊙ tanh(c_t)"]

        xt --> f_t
        xt --> i_t
        xt --> c_tilde
        xt --> o_t

        h_prev --> f_t
        h_prev --> i_t
        h_prev --> c_tilde
        h_prev --> o_t

        c_prev --> c_update
        f_t --> c_update
        i_t --> c_update
        c_tilde --> c_update

        c_update --> h_update
        o_t --> h_update
        
        c_update --> c_out["Salida Celda: c_t"]
        h_update --> h_out["Salida Oculta: h_t"]
    end
```

---

### Citas Bibliográficas Científicas

* **Hochreiter, S., & Schmidhuber, J. (1997).** *Long Short-Term Memory*. Neural Computation, 9(8), 1735-1780. [Enlace al Paper](https://doi.org/10.1162/neco.1997.9.8.1735)
* **Gers, F. A., Schmidhuber, J., & Cummins, F. (2000).** *Learning to forget: Continual prediction with LSTM*. Neural Computation, 12(10), 2451-2471. [Enlace al Paper](https://doi.org/10.1162/089976600300015015) *(Introduce la compuerta de olvido original en la formulación clásica de LSTM)*.

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomLSTMRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

KeyboardInterrupt: 

## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos crudos (ECG, EDA, Resp, EEG) desde el dataset `fatigueset`, alineamos los streams temporales mediante `merge_asof` e interpolamos valores perdidos para construir tencesor de secuencias temporales coherentes.

In [ ]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


Preparando targets del dataframe ML...


Combinando streams fisiológicos crudos (Chest y Wrist)...


Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split) y DataLoaders

Para garantizar la rigurosidad científica y evitar la fuga de información (data leakage), separamos el dataset de tal manera que el sujeto utilizado para validar/probar nunca esté presente en el conjunto de entrenamiento.

In [ ]:
# Hacemos una partición de prueba donde el participante '01' se reserva para validación
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor LSTM Manual

Instanciamos nuestro modelo regresor `CustomLSTMRegressor` usando el número de características predictivas fisiológicas detectadas. El modelo predice simultáneamente los niveles continuos de `fatiga_fisica` y `fatiga_mental` (salida bidimensional).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
hidden_size = 64
num_layers = 2
dropout = 0.2

model = CustomLSTMRegressor(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

CustomLSTMRegressor(
  (lstm): CustomLSTM(
    (layers): ModuleList(
      (0-1): 2 x CustomLSTMCell()
    )
    (dropout_layer): Dropout(p=0.2, inplace=False)
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 5. Entrenamiento Corto de Validación (Sanity Check)

Ejecutamos un entrenamiento corto de 2 épocas sobre el conjunto de datos para verificar la estabilidad de las operaciones de tensores en las celdas LSTM y del motor común de optimización.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando mini-entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        # Gradient clipping para evitar inestabilidad recurrente
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento corto finalizado correctamente.")

Iniciando mini-entrenamiento...


Epoch 1/2 - Train Loss (MSE): 1255.959692 - Val Loss (MSE): 1143.591675


Epoch 2/2 - Train Loss (MSE): 1044.125594 - Val Loss (MSE): 946.136078
[OK] Entrenamiento corto finalizado correctamente.


## 6. Serialización del Modelo de Fatiga

Persistimos el estado del modelo en la carpeta centralizada de modelos del proyecto `/models/` bajo la categoría correspondiente.

In [ ]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "lstm_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\lstm_fatigue_notebook.pt
